# Bài 03.2 — Dịch máy Anh ↔ Việt với Bahdanau Attention

Notebook xây dựng **hai mô hình Encoder–Decoder LSTM độc lập**:

1. English → Vietnamese (`en_to_vi`)
2. Vietnamese → English (`vi_to_en`)

Dữ liệu: [IWSLT'15 English–Vietnamese trên Kaggle](https://www.kaggle.com/datasets/tuannguyenvananh/iwslt15-englishvietnamesetrung).

Decoder dùng **Bahdanau additive attention** ở từng bước sinh token, có source padding mask. Notebook còn trực quan hóa ma trận attention, đánh giá SacreBLEU theo nhóm độ dài câu và đọc benchmark của mô hình không Attention để so sánh công bằng.

## 1. Cài đặt thư viện

Cell dưới cài đầy đủ các package được notebook sử dụng. Gói `torch` đã bao gồm
`torch.nn`, `torch.utils.data`, `torch.nn.utils` và các lớp LSTM/DataLoader nên
không cần cài chúng thành package riêng. Không dùng `-U` cho PyTorch để tránh
ghi đè bản CUDA tương thích đã có sẵn trên Colab hoặc Kaggle.

`torchvision`, `torchaudio` và `torchtext` không được cài vì notebook không
xử lý ảnh/âm thanh và không import các package này.

In [ ]:
%pip install -q torch
%pip install -q kagglehub sacrebleu numpy pandas matplotlib tqdm

## 2. Import

In [ ]:
import html
import math
import os
import random
import re
import shutil
import tarfile
import time
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path
from typing import Iterable, Sequence

import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

# Import PyTorch before NumPy/Pandas/Matplotlib to avoid a Windows DLL/OpenMP
# loading conflict in a few local Python distributions.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sacrebleu.metrics import BLEU
from tqdm.auto import tqdm

## 3. Cấu hình thí nghiệm

- Đặt `QUICK_RUN = True` để kiểm tra nhanh pipeline trên một phần dữ liệu.
- Giữ `QUICK_RUN = False` cho bài chạy chính thức với toàn bộ tập train sau làm sạch.
- Câu dài hơn `MAX_SENTENCE_TOKENS` bị loại để kiểm soát bộ nhớ và giữ đúng cùng tập dữ liệu với baseline.

In [ ]:
SEED = 42
DATASET_SLUG = "tuannguyenvananh/iwslt15-englishvietnamesetrung"

QUICK_RUN = False
QUICK_TRAIN_PAIRS = 20_000
QUICK_EVAL_PAIRS = 300

MAX_SENTENCE_TOKENS = 50
MAX_LENGTH_RATIO = 4.0
MIN_TOKEN_FREQUENCY = 2
MAX_VOCAB_SIZE = 20_000  # Bao gồm bốn special tokens.

BATCH_SIZE = 64
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.30
LEARNING_RATE = 1e-3
EPOCHS = 2 if QUICK_RUN else 100
TEACHER_FORCING_RATIO = 0.50
GRADIENT_CLIP = 1.0
EARLY_STOPPING_PATIENCE = 3
MAX_DECODE_LENGTH = 60
NUM_WORKERS = 0

if Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working/en_vi_seq2seq_bahdanau")
elif Path("/content").exists():
    BASE_DIR = Path("/content/en_vi_seq2seq_bahdanau")
else:
    BASE_DIR = Path.cwd() / "en_vi_seq2seq_bahdanau"

DATA_DIR = BASE_DIR / "data"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
BENCHMARK_DIR = BASE_DIR.parent / "en_vi_translation_benchmarks"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
print(f"Artifacts: {BASE_DIR.resolve()}")
print(f"Quick run: {QUICK_RUN} | epochs: {EPOCHS}")

## 4. Cố định seed

Cùng seed được dùng cho Python, NumPy, PyTorch và thứ tự shuffle của DataLoader.

In [ ]:
def set_seed(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed()

## 5. Tìm hoặc tải IWSLT'15

Thứ tự ưu tiên:

1. Dataset đã được gắn tại `/kaggle/input/...` hoặc đã có trong thư mục làm việc.
2. Tải đúng Kaggle dataset bằng `kagglehub`.
3. Nếu Kaggle yêu cầu đăng nhập/consent, tải mirror của đúng sáu file IWSLT'15 gốc (`train`, `tst2012`, `tst2013`).

Nhờ vậy notebook chạy được trên Kaggle, Colab và máy cá nhân. Loader luôn kiểm tra đủ sáu file trước khi tiếp tục.

In [ ]:
REQUIRED_FILENAMES = (
    "train.en",
    "train.vi",
    "tst2012.en",
    "tst2012.vi",
    "tst2013.en",
    "tst2013.vi",
)

MIRROR_ARCHIVES = {
    "train-en-vi.tgz": (
        "https://raw.githubusercontent.com/stefan-it/nmt-en-vi/master/"
        "data/train-en-vi.tgz"
    ),
    "dev-2012-en-vi.tgz": (
        "https://raw.githubusercontent.com/stefan-it/nmt-en-vi/master/"
        "data/dev-2012-en-vi.tgz"
    ),
    "test-2013-en-vi.tgz": (
        "https://raw.githubusercontent.com/stefan-it/nmt-en-vi/master/"
        "data/test-2013-en-vi.tgz"
    ),
}


def locate_required_files(root: Path) -> dict[str, Path] | None:
    """Tìm đủ sáu file theo basename trong một thư mục và các thư mục con."""
    if not root.exists() or not root.is_dir():
        return None

    found: dict[str, Path] = {}
    for path in root.rglob("*"):
        if path.is_file() and path.name in REQUIRED_FILENAMES:
            found.setdefault(path.name, path)
    return found if all(name in found for name in REQUIRED_FILENAMES) else None


def download_iwslt_mirror(destination: Path) -> dict[str, Path]:
    """Tải và giải nén an toàn mirror của các file IWSLT'15 gốc."""
    destination.mkdir(parents=True, exist_ok=True)

    for archive_name, url in MIRROR_ARCHIVES.items():
        archive_path = destination / archive_name
        if not archive_path.exists():
            print(f"Downloading mirror: {archive_name}")
            temporary_path = archive_path.with_suffix(archive_path.suffix + ".part")
            urllib.request.urlretrieve(url, temporary_path)
            temporary_path.replace(archive_path)

        with tarfile.open(archive_path, mode="r:gz") as archive:
            for member in archive.getmembers():
                basename = Path(member.name).name
                if not member.isfile() or basename not in REQUIRED_FILENAMES:
                    continue
                output_path = destination / basename
                if output_path.exists():
                    continue
                source = archive.extractfile(member)
                if source is None:
                    raise RuntimeError(f"Cannot read {member.name} from {archive_path}")
                with source, output_path.open("wb") as target:
                    shutil.copyfileobj(source, target)

    files = locate_required_files(destination)
    if files is None:
        raise FileNotFoundError("Mirror download completed but required files are missing.")
    return files


def resolve_dataset_files() -> tuple[dict[str, Path], str]:
    dataset_folder_name = DATASET_SLUG.split("/")[-1]
    candidate_roots = [
        Path("/kaggle/input") / dataset_folder_name,
        Path("/kaggle/input"),
        DATA_DIR,
        Path.cwd() / "data",
    ]
    if Path.cwd() != Path(Path.cwd().anchor):
        candidate_roots.append(Path.cwd())

    for root in candidate_roots:
        files = locate_required_files(root)
        if files is not None:
            return files, f"local/Kaggle mount: {root}"

    try:
        import kagglehub

        downloaded_path = Path(kagglehub.dataset_download(DATASET_SLUG))
        files = locate_required_files(downloaded_path)
        if files is not None:
            return files, f"kagglehub: {downloaded_path}"
        print("Kaggle download succeeded, but the expected six files were not found.")
    except Exception as error:
        print(f"Kaggle download unavailable: {type(error).__name__}: {error}")

    files = download_iwslt_mirror(DATA_DIR)
    return files, f"IWSLT'15 mirror: {DATA_DIR}"


dataset_files, dataset_source = resolve_dataset_files()
print(f"Dataset source: {dataset_source}")

## 6. Kiểm tra file và số dòng song song

Mỗi split phải có số dòng tiếng Anh bằng số dòng tiếng Việt. Không dùng `zip` để âm thầm bỏ dòng dư; nếu lệch, notebook dừng bằng lỗi rõ ràng.

In [ ]:
def count_lines(path: Path) -> int:
    with path.open("r", encoding="utf-8", errors="strict") as file:
        return sum(1 for _ in file)


inventory_rows = []
for filename in REQUIRED_FILENAMES:
    path = dataset_files[filename]
    inventory_rows.append(
        {
            "file": filename,
            "size_MB": path.stat().st_size / (1024**2),
            "number_of_lines": count_lines(path),
            "path": str(path),
        }
    )

inventory_df = pd.DataFrame(inventory_rows)
display(inventory_df.style.format({"size_MB": "{:.3f}"}))

for split_name in ("train", "tst2012", "tst2013"):
    en_count = int(
        inventory_df.loc[inventory_df["file"] == f"{split_name}.en", "number_of_lines"].iloc[0]
    )
    vi_count = int(
        inventory_df.loc[inventory_df["file"] == f"{split_name}.vi", "number_of_lines"].iloc[0]
    )
    if en_count != vi_count:
        raise ValueError(f"{split_name}: {en_count:,} EN lines != {vi_count:,} VI lines")
    print(f"{split_name:8s}: {en_count:,} aligned pairs")

In [ ]:
def read_utf8_lines(path: Path) -> list[str]:
    return path.read_text(encoding="utf-8", errors="strict").splitlines()


raw_lines: dict[str, dict[str, list[str]]] = {}
for split_name in ("train", "tst2012", "tst2013"):
    raw_lines[split_name] = {
        "en": read_utf8_lines(dataset_files[f"{split_name}.en"]),
        "vi": read_utf8_lines(dataset_files[f"{split_name}.vi"]),
    }

raw_preview = pd.DataFrame(
    {
        "English": raw_lines["train"]["en"][:5],
        "Vietnamese": raw_lines["train"]["vi"][:5],
    }
)
display(raw_preview)

## 7. Chuẩn hóa và tokenize riêng cho tiếng Anh/tiếng Việt

Hai tokenizer là hai hàm độc lập:

- English tokenizer giữ contraction như `don't` thành một token.
- Vietnamese tokenizer hỗ trợ ký tự Unicode có dấu và token ghép bằng `_`.

IWSLT'15 đã có khoảng trắng quanh phần lớn dấu câu. Regex vẫn được áp dụng lại để cách xử lý câu nhập mới nhất quán với dữ liệu huấn luyện. Không xóa dấu tiếng Việt.

In [ ]:
WHITESPACE_RE = re.compile(r"\s+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
CONTROL_CHARACTER_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
ENGLISH_TOKEN_RE = re.compile(r"\w+(?:['’]\w+)?|[^\w\s]", flags=re.UNICODE)
VIETNAMESE_TOKEN_RE = re.compile(r"\w+(?:[_'’]\w+)*|[^\w\s]", flags=re.UNICODE)


def normalize_text(text: str, lowercase: bool = True) -> str:
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("##AT##-##AT##", "-").replace("\u00a0", " ")
    text = HTML_TAG_RE.sub(" ", text)
    text = CONTROL_CHARACTER_RE.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text.lower() if lowercase else text


def tokenize_english(text: str) -> list[str]:
    """Tokenizer riêng cho tiếng Anh."""
    # IWSLT/Moses có thể lưu "I &apos;d"; sau html.unescape nó thành
    # "I 'd". Ghép lại để câu trong corpus và câu người dùng được xử lý giống nhau.
    text = re.sub(
        r"\b(\w+)\s+(['’](?:s|re|ve|ll|d|m|t))\b",
        r"\1\2",
        text,
        flags=re.IGNORECASE,
    )
    return ENGLISH_TOKEN_RE.findall(text)


def tokenize_vietnamese(text: str) -> list[str]:
    """Tokenizer riêng cho tiếng Việt; giữ nguyên dấu và dấu gạch dưới."""
    return VIETNAMESE_TOKEN_RE.findall(text)


TOKENIZERS = {"en": tokenize_english, "vi": tokenize_vietnamese}

tokenizer_demo = pd.DataFrame(
    [
        {
            "language": "en",
            "sentence": "I don't remove punctuation, and I'm ready!",
            "tokens": tokenize_english(normalize_text("I don't remove punctuation, and I'm ready!")),
        },
        {
            "language": "vi",
            "sentence": "Tôi đang học xử_lý ngôn ngữ tự nhiên!",
            "tokens": tokenize_vietnamese(normalize_text("Tôi đang học xử_lý ngôn ngữ tự nhiên!")),
        },
    ]
)
display(tokenizer_demo)

## 8. Làm sạch từng cặp câu

Một cặp bị loại nếu rỗng, chứa ký tự thay thế Unicode, quá dài, chênh lệch độ dài bất thường hoặc trùng hoàn toàn trong cùng split. Cell trả về thống kê cụ thể thay vì làm sạch âm thầm.

In [ ]:
def clean_parallel_split(
    split_name: str,
    english_lines: Sequence[str],
    vietnamese_lines: Sequence[str],
) -> tuple[pd.DataFrame, Counter]:
    if len(english_lines) != len(vietnamese_lines):
        raise ValueError(
            f"{split_name}: {len(english_lines):,} English lines != "
            f"{len(vietnamese_lines):,} Vietnamese lines"
        )

    statistics = Counter(raw_pairs=len(english_lines))
    records = []
    seen_pairs: set[tuple[str, str]] = set()

    for line_number, (english_raw, vietnamese_raw) in enumerate(
        zip(english_lines, vietnamese_lines), start=1
    ):
        english = normalize_text(english_raw)
        vietnamese = normalize_text(vietnamese_raw)

        if not english or not vietnamese:
            statistics["removed_empty"] += 1
            continue
        if "\ufffd" in english or "\ufffd" in vietnamese:
            statistics["removed_unicode_replacement"] += 1
            continue

        english_tokens = tokenize_english(english)
        vietnamese_tokens = tokenize_vietnamese(vietnamese)
        if not english_tokens or not vietnamese_tokens:
            statistics["removed_no_tokens"] += 1
            continue
        if max(len(english_tokens), len(vietnamese_tokens)) > MAX_SENTENCE_TOKENS:
            statistics["removed_too_long"] += 1
            continue

        length_ratio = max(len(english_tokens), len(vietnamese_tokens)) / min(
            len(english_tokens), len(vietnamese_tokens)
        )
        if length_ratio > MAX_LENGTH_RATIO:
            statistics["removed_length_ratio"] += 1
            continue

        pair_key = (" ".join(english_tokens), " ".join(vietnamese_tokens))
        if pair_key in seen_pairs:
            statistics["removed_duplicate"] += 1
            continue
        seen_pairs.add(pair_key)

        if english == vietnamese:
            statistics["kept_identical_text"] += 1

        records.append(
            {
                "line_number": line_number,
                "en": english,
                "vi": vietnamese,
                "en_tokens": english_tokens,
                "vi_tokens": vietnamese_tokens,
                "en_length": len(english_tokens),
                "vi_length": len(vietnamese_tokens),
            }
        )
        statistics["kept"] += 1

    frame = pd.DataFrame.from_records(records)
    if frame.empty:
        raise ValueError(f"No valid sentence pairs remain in {split_name}.")
    return frame, statistics


split_mapping = {"train": "train", "validation": "tst2012", "test": "tst2013"}
clean_frames: dict[str, pd.DataFrame] = {}
cleaning_statistics = []

for logical_split, file_split in split_mapping.items():
    frame, statistics = clean_parallel_split(
        logical_split,
        raw_lines[file_split]["en"],
        raw_lines[file_split]["vi"],
    )
    clean_frames[logical_split] = frame
    cleaning_statistics.append({"split": logical_split, **dict(statistics)})

cleaning_report_df = pd.DataFrame(cleaning_statistics).fillna(0)
numeric_columns = cleaning_report_df.columns.drop("split")
cleaning_report_df[numeric_columns] = cleaning_report_df[numeric_columns].astype(int)
display(cleaning_report_df)

## 9. Loại rò rỉ giữa train/validation/test

Các split chính thức vẫn được giữ nguyên vai trò. Nếu một cặp trùng xuất hiện ở split sau, cặp đó bị loại khỏi split sau. Vocabulary chỉ được tạo từ `train`.

In [ ]:
def frame_pair_keys(frame: pd.DataFrame) -> list[tuple[str, str]]:
    return list(zip(frame["en"], frame["vi"]))


def remove_overlap(
    frame: pd.DataFrame,
    forbidden_pairs: set[tuple[str, str]],
    split_name: str,
) -> tuple[pd.DataFrame, int]:
    keys = frame_pair_keys(frame)
    keep_mask = [key not in forbidden_pairs for key in keys]
    removed = len(keep_mask) - sum(keep_mask)
    cleaned = frame.loc[keep_mask].reset_index(drop=True)
    print(f"{split_name}: removed {removed:,} pairs overlapping earlier splits")
    return cleaned, removed


train_full_df = clean_frames["train"].reset_index(drop=True)
train_pair_set = set(frame_pair_keys(train_full_df))

validation_full_df, _ = remove_overlap(
    clean_frames["validation"], train_pair_set, "validation"
)
validation_pair_set = set(frame_pair_keys(validation_full_df))

test_full_df, _ = remove_overlap(
    clean_frames["test"], train_pair_set | validation_pair_set, "test"
)

if QUICK_RUN:
    train_df = (
        train_full_df.sample(
            n=min(QUICK_TRAIN_PAIRS, len(train_full_df)), random_state=SEED
        )
        .sort_values("line_number")
        .reset_index(drop=True)
    )
    validation_df = validation_full_df.head(QUICK_EVAL_PAIRS).reset_index(drop=True)
    test_df = test_full_df.head(QUICK_EVAL_PAIRS).reset_index(drop=True)
else:
    train_df = train_full_df.copy()
    validation_df = validation_full_df.copy()
    test_df = test_full_df.copy()

assert set(frame_pair_keys(train_df)).isdisjoint(frame_pair_keys(validation_df))
assert set(frame_pair_keys(train_df)).isdisjoint(frame_pair_keys(test_df))
assert set(frame_pair_keys(validation_df)).isdisjoint(frame_pair_keys(test_df))

split_frames = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

print(
    f"Used pairs — train: {len(train_df):,}, validation: {len(validation_df):,}, "
    f"test: {len(test_df):,}"
)

## 10. Thống kê sau làm sạch

In [ ]:
def summarize_split(name: str, frame: pd.DataFrame) -> dict[str, float | int | str]:
    return {
        "split": name,
        "pairs": len(frame),
        "mean_en_length": frame["en_length"].mean(),
        "p95_en_length": frame["en_length"].quantile(0.95),
        "max_en_length": frame["en_length"].max(),
        "mean_vi_length": frame["vi_length"].mean(),
        "p95_vi_length": frame["vi_length"].quantile(0.95),
        "max_vi_length": frame["vi_length"].max(),
    }


split_summary_df = pd.DataFrame(
    [summarize_split(name, frame) for name, frame in split_frames.items()]
)
display(
    split_summary_df.style.format(
        {
            "mean_en_length": "{:.2f}",
            "p95_en_length": "{:.1f}",
            "mean_vi_length": "{:.2f}",
            "p95_vi_length": "{:.1f}",
        }
    )
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for axis, column, title in (
    (axes[0], "en_length", "English token lengths"),
    (axes[1], "vi_length", "Vietnamese token lengths"),
):
    axis.hist(train_df[column], bins=30, alpha=0.85, color="#4C78A8")
    axis.set_title(title)
    axis.set_xlabel("Number of tokens")
    axis.set_ylabel("Sentence pairs")
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 11. Tạo vocabulary từ tập train

Thứ tự ID của special tokens được cố định:

- `<PAD>` = 0
- `<SOS>` = 1
- `<EOS>` = 2
- `<UNK>` = 3

Token validation/test không có trong train được ánh xạ sang `<UNK>` để tránh data leakage.

In [ ]:
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]


class Vocabulary:
    def __init__(
        self,
        token_sequences: Iterable[Sequence[str]],
        min_frequency: int = 1,
        max_size: int | None = None,
    ) -> None:
        self.counter = Counter(token for sequence in token_sequences for token in sequence)
        candidates = [
            (token, frequency)
            for token, frequency in self.counter.items()
            if frequency >= min_frequency and token not in SPECIAL_TOKENS
        ]
        candidates.sort(key=lambda item: (-item[1], item[0]))

        if max_size is not None:
            candidates = candidates[: max(0, max_size - len(SPECIAL_TOKENS))]

        self.id_to_token = SPECIAL_TOKENS + [token for token, _ in candidates]
        self.token_to_id = {
            token: token_id for token_id, token in enumerate(self.id_to_token)
        }

        self.pad_id = self.token_to_id[PAD_TOKEN]
        self.sos_id = self.token_to_id[SOS_TOKEN]
        self.eos_id = self.token_to_id[EOS_TOKEN]
        self.unk_id = self.token_to_id[UNK_TOKEN]

    def __len__(self) -> int:
        return len(self.id_to_token)

    def encode(
        self, tokens: Sequence[str], add_boundary_tokens: bool = True
    ) -> list[int]:
        token_ids = [self.token_to_id.get(token, self.unk_id) for token in tokens]
        if add_boundary_tokens:
            token_ids = [self.sos_id, *token_ids, self.eos_id]
        return token_ids

    def decode(
        self,
        token_ids: Sequence[int],
        stop_at_eos: bool = True,
        skip_special_tokens: bool = True,
    ) -> list[str]:
        tokens = []
        for token_id in token_ids:
            token = self.id_to_token[int(token_id)]
            if stop_at_eos and token == EOS_TOKEN:
                break
            # Giữ <UNK> trong output: đây là một dự đoán có ý nghĩa và phải bị
            # BLEU phạt. Chỉ bỏ padding/boundary tokens.
            if skip_special_tokens and token in {PAD_TOKEN, SOS_TOKEN, EOS_TOKEN}:
                continue
            tokens.append(token)
        return tokens


english_vocab = Vocabulary(
    train_df["en_tokens"],
    min_frequency=MIN_TOKEN_FREQUENCY,
    max_size=MAX_VOCAB_SIZE,
)
vietnamese_vocab = Vocabulary(
    train_df["vi_tokens"],
    min_frequency=MIN_TOKEN_FREQUENCY,
    max_size=MAX_VOCAB_SIZE,
)
VOCABS = {"en": english_vocab, "vi": vietnamese_vocab}

for language, vocabulary in VOCABS.items():
    assert vocabulary.id_to_token[:4] == SPECIAL_TOKENS
    print(
        f"{language.upper()} vocabulary: {len(vocabulary):,} / "
        f"{len(vocabulary.counter):,} observed unique tokens"
    )
    print("  12 most frequent:", vocabulary.counter.most_common(12))

In [ ]:
def unknown_rate(frame: pd.DataFrame, language: str, vocabulary: Vocabulary) -> float:
    token_column = f"{language}_tokens"
    total = 0
    unknown = 0
    for tokens in frame[token_column]:
        total += len(tokens)
        unknown += sum(token not in vocabulary.token_to_id for token in tokens)
    return unknown / max(total, 1)


coverage_rows = []
for split_name, frame in split_frames.items():
    coverage_rows.append(
        {
            "split": split_name,
            "English_UNK_rate": unknown_rate(frame, "en", english_vocab),
            "Vietnamese_UNK_rate": unknown_rate(frame, "vi", vietnamese_vocab),
        }
    )
display(pd.DataFrame(coverage_rows).style.format({"English_UNK_rate": "{:.2%}", "Vietnamese_UNK_rate": "{:.2%}"}))

## 12. Dataset, padding động và DataLoader

Mỗi câu được thêm `<SOS>` và `<EOS>`. `pad_sequence` chỉ pad đến câu dài nhất trong batch, bằng ID của `<PAD>`. Source lengths được giữ lại để Encoder dùng packed sequence và bỏ qua padding.

In [ ]:
DIRECTION_SPECS = {
    "en_to_vi": {
        "label": "English → Vietnamese",
        "source_language": "en",
        "target_language": "vi",
        "source_column": "en_tokens",
        "target_column": "vi_tokens",
    },
    "vi_to_en": {
        "label": "Vietnamese → English",
        "source_language": "vi",
        "target_language": "en",
        "source_column": "vi_tokens",
        "target_column": "en_tokens",
    },
}


class ParallelTranslationDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        source_column: str,
        target_column: str,
        source_vocab: Vocabulary,
        target_vocab: Vocabulary,
    ) -> None:
        self.frame = frame.reset_index(drop=True)
        self.source_column = source_column
        self.target_column = target_column
        self.source_vocab = source_vocab
        self.target_vocab = target_vocab

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.frame.iloc[index]
        source_ids = self.source_vocab.encode(row[self.source_column])
        target_ids = self.target_vocab.encode(row[self.target_column])
        return (
            torch.tensor(source_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long),
        )


def make_collate_fn(source_pad_id: int, target_pad_id: int):
    def collate_batch(batch: list[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
        source_sequences, target_sequences = zip(*batch)
        source_lengths = torch.tensor(
            [len(sequence) for sequence in source_sequences], dtype=torch.long
        )
        source_batch = pad_sequence(
            source_sequences, batch_first=True, padding_value=source_pad_id
        )
        target_batch = pad_sequence(
            target_sequences, batch_first=True, padding_value=target_pad_id
        )
        return {
            "source": source_batch,
            "source_lengths": source_lengths,
            "target": target_batch,
        }

    return collate_batch


def create_direction_data(direction: str) -> dict:
    spec = DIRECTION_SPECS[direction]
    source_vocab = VOCABS[spec["source_language"]]
    target_vocab = VOCABS[spec["target_language"]]
    collate_fn = make_collate_fn(source_vocab.pad_id, target_vocab.pad_id)

    datasets = {
        split_name: ParallelTranslationDataset(
            frame,
            spec["source_column"],
            spec["target_column"],
            source_vocab,
            target_vocab,
        )
        for split_name, frame in split_frames.items()
    }

    generator = torch.Generator().manual_seed(SEED)
    common = {
        "batch_size": BATCH_SIZE,
        "collate_fn": collate_fn,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
        "persistent_workers": NUM_WORKERS > 0,
    }
    loaders = {
        "train": DataLoader(
            datasets["train"], shuffle=True, generator=generator, **common
        ),
        "validation": DataLoader(datasets["validation"], shuffle=False, **common),
        "test": DataLoader(datasets["test"], shuffle=False, **common),
    }
    return {
        "spec": spec,
        "source_vocab": source_vocab,
        "target_vocab": target_vocab,
        "datasets": datasets,
        "loaders": loaders,
    }


direction_data = {
    direction: create_direction_data(direction) for direction in DIRECTION_SPECS
}

## 13. Kiểm tra một batch đã padding

In [ ]:
for direction, data_bundle in direction_data.items():
    batch = next(iter(data_bundle["loaders"]["train"]))
    source_vocab = data_bundle["source_vocab"]
    target_vocab = data_bundle["target_vocab"]
    print(f"\n{data_bundle['spec']['label']}")
    print("source shape:", tuple(batch["source"].shape))
    print("target shape:", tuple(batch["target"].shape))
    print("source lengths:", batch["source_lengths"][:8].tolist())
    print("source PAD id:", source_vocab.pad_id, "| target PAD id:", target_vocab.pad_id)
    print("decoded source[0]:", source_vocab.decode(batch["source"][0].tolist())[:30])
    print("decoded target[0]:", target_vocab.decode(batch["target"][0].tolist())[:30])

## 14. Encoder–Decoder LSTM với Bahdanau additive attention

Encoder trả về toàn bộ hidden states. Tại mỗi bước, Bahdanau attention tính
$e_{t,s}=v^T\tanh(W_qh_t+W_kh_s)$, mask vị trí `<PAD>`, rồi softmax theo trục source.
Context vector là tổng có trọng số của các encoder states và được đưa vào Decoder.

In [ ]:
class EncoderLSTM(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        embedding_dim: int,
        hidden_dim: int,
        num_layers: int,
        dropout: float,
        pad_id: int,
    ) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(
            vocabulary_size, embedding_dim, padding_idx=pad_id
        )
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )

    def forward(
        self, source: torch.Tensor, source_lengths: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        embedded = self.dropout(self.embedding(source))
        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, (hidden, cell) = self.lstm(packed)
        encoder_outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=source.size(1),
        )
        return encoder_outputs, hidden, cell


class BahdanauAttention(nn.Module):
    """Additive attention: learned non-linear compatibility score."""

    def __init__(self, hidden_dim: int, attention_dim: int | None = None) -> None:
        super().__init__()
        attention_dim = attention_dim or hidden_dim
        self.query_projection = nn.Linear(hidden_dim, attention_dim, bias=False)
        self.key_projection = nn.Linear(hidden_dim, attention_dim, bias=False)
        self.energy_projection = nn.Linear(attention_dim, 1, bias=False)

    def forward(
        self,
        query: torch.Tensor,
        encoder_outputs: torch.Tensor,
        source_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        projected_query = self.query_projection(query).unsqueeze(1)
        projected_keys = self.key_projection(encoder_outputs)
        scores = self.energy_projection(
            torch.tanh(projected_query + projected_keys)
        ).squeeze(-1)
        scores = scores.masked_fill(~source_mask, torch.finfo(scores.dtype).min)
        attention_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(
            attention_weights.unsqueeze(1), encoder_outputs
        ).squeeze(1)
        return context, attention_weights


class BahdanauDecoderLSTM(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        embedding_dim: int,
        hidden_dim: int,
        num_layers: int,
        dropout: float,
        pad_id: int,
    ) -> None:
        super().__init__()
        self.output_dim = vocabulary_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(
            vocabulary_size, embedding_dim, padding_idx=pad_id
        )
        self.dropout = nn.Dropout(dropout)
        self.attention = BahdanauAttention(hidden_dim)
        self.lstm = nn.LSTM(
            embedding_dim + hidden_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.output_projection = nn.Linear(
            hidden_dim * 2 + embedding_dim, vocabulary_size
        )

    def forward(
        self,
        input_token: torch.Tensor,
        hidden: torch.Tensor,
        cell: torch.Tensor,
        encoder_outputs: torch.Tensor,
        source_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        embedded = self.dropout(self.embedding(input_token))
        query = hidden[-1]
        context, attention_weights = self.attention(
            query, encoder_outputs, source_mask
        )
        decoder_input = torch.cat([embedded, context], dim=1).unsqueeze(1)
        output, (hidden, cell) = self.lstm(decoder_input, (hidden, cell))
        output = output.squeeze(1)
        logits = self.output_projection(torch.cat([output, context, embedded], dim=1))
        return logits, hidden, cell, attention_weights


class Seq2SeqBahdanau(nn.Module):
    def __init__(
        self,
        encoder: EncoderLSTM,
        decoder: BahdanauDecoderLSTM,
        source_pad_id: int,
    ) -> None:
        super().__init__()
        if encoder.hidden_dim != decoder.hidden_dim:
            raise ValueError("Encoder and decoder hidden dimensions must match.")
        if encoder.num_layers != decoder.num_layers:
            raise ValueError("Encoder and decoder layer counts must match.")
        self.encoder = encoder
        self.decoder = decoder
        self.source_pad_id = source_pad_id

    def forward(
        self,
        source: torch.Tensor,
        source_lengths: torch.Tensor,
        target: torch.Tensor,
        teacher_forcing_ratio: float = 0.5,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size, target_length = target.shape
        source_length = source.size(1)
        outputs = torch.zeros(
            batch_size,
            target_length,
            self.decoder.output_dim,
            device=source.device,
        )
        attention_matrices = torch.zeros(
            batch_size, target_length, source_length, device=source.device
        )

        encoder_outputs, hidden, cell = self.encoder(source, source_lengths)
        source_mask = source.ne(self.source_pad_id)
        input_token = target[:, 0]  # <SOS>

        for time_step in range(1, target_length):
            logits, hidden, cell, attention_weights = self.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs,
                source_mask,
            )
            outputs[:, time_step] = logits
            attention_matrices[:, time_step] = attention_weights
            greedy_token = logits.argmax(dim=1)
            use_teacher = random.random() < teacher_forcing_ratio
            input_token = target[:, time_step] if use_teacher else greedy_token

        return outputs, attention_matrices


def initialize_model(model: nn.Module) -> None:
    for parameter in model.parameters():
        if parameter.dim() > 1:
            nn.init.xavier_uniform_(parameter)
        else:
            nn.init.zeros_(parameter)

    with torch.no_grad():
        model.encoder.embedding.weight[model.encoder.embedding.padding_idx].zero_()
        model.decoder.embedding.weight[model.decoder.embedding.padding_idx].zero_()


def build_model(direction: str) -> Seq2SeqBahdanau:
    data_bundle = direction_data[direction]
    source_vocab = data_bundle["source_vocab"]
    target_vocab = data_bundle["target_vocab"]
    encoder = EncoderLSTM(
        len(source_vocab),
        EMBEDDING_DIM,
        HIDDEN_DIM,
        NUM_LAYERS,
        DROPOUT,
        source_vocab.pad_id,
    )
    decoder = BahdanauDecoderLSTM(
        len(target_vocab),
        EMBEDDING_DIM,
        HIDDEN_DIM,
        NUM_LAYERS,
        DROPOUT,
        target_vocab.pad_id,
    )
    model = Seq2SeqBahdanau(encoder, decoder, source_vocab.pad_id)
    initialize_model(model)
    return model


for direction, spec in DIRECTION_SPECS.items():
    temporary_model = build_model(direction)
    number_of_parameters = sum(p.numel() for p in temporary_model.parameters())
    print(f"{spec['label']}: {number_of_parameters:,} trainable parameters")
    del temporary_model

## 15. Teacher Forcing, loss và vòng lặp huấn luyện

Ở train, với xác suất `TEACHER_FORCING_RATIO`, token đúng tại bước hiện tại được đưa vào Decoder ở bước tiếp theo. Ở validation, ratio bằng 0 để phản ánh suy luận tự hồi quy. Loss và token accuracy đều bỏ qua `<PAD>`.

In [ ]:
def run_epoch(
    model: Seq2SeqBahdanau,
    dataloader: DataLoader,
    criterion: nn.Module,
    target_pad_id: int,
    optimizer: torch.optim.Optimizer | None = None,
    teacher_forcing_ratio: float = 0.0,
) -> dict[str, float]:
    is_training = optimizer is not None
    model.train(is_training)

    total_weighted_loss = 0.0
    total_tokens = 0
    total_correct = 0

    progress = tqdm(
        dataloader,
        desc="train" if is_training else "validation",
        leave=False,
    )
    for batch in progress:
        source = batch["source"].to(DEVICE, non_blocking=True)
        source_lengths = batch["source_lengths"]
        target = batch["target"].to(DEVICE, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits, _ = model(
                source,
                source_lengths,
                target,
                teacher_forcing_ratio=teacher_forcing_ratio,
            )
            prediction_logits = logits[:, 1:].reshape(-1, logits.size(-1))
            target_tokens = target[:, 1:].reshape(-1)
            loss = criterion(prediction_logits, target_tokens)

            if is_training:
                loss.backward()
                clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                optimizer.step()

        non_padding_mask = target_tokens.ne(target_pad_id)
        real_token_count = int(non_padding_mask.sum().item())
        predictions = prediction_logits.argmax(dim=1)
        correct = int(
            predictions.eq(target_tokens).logical_and(non_padding_mask).sum().item()
        )

        total_weighted_loss += loss.item() * real_token_count
        total_tokens += real_token_count
        total_correct += correct
        progress.set_postfix(loss=f"{loss.item():.4f}")

    mean_loss = total_weighted_loss / max(total_tokens, 1)
    return {
        "loss": mean_loss,
        "perplexity": math.exp(min(mean_loss, 20.0)),
        "token_accuracy": total_correct / max(total_tokens, 1),
    }


def load_local_checkpoint(path: Path) -> dict:
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:  # Tương thích các bản PyTorch cũ.
        return torch.load(path, map_location="cpu")


def train_direction(direction: str) -> tuple[Seq2SeqBahdanau, dict[str, list[float]], Path]:
    set_seed(SEED)
    data_bundle = direction_data[direction]
    target_vocab = data_bundle["target_vocab"]
    model = build_model(direction).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=target_vocab.pad_id)

    history = {
        "train_loss": [],
        "validation_loss": [],
        "train_token_accuracy": [],
        "validation_token_accuracy": [],
        "epoch_seconds": [],
    }
    checkpoint_path = CHECKPOINT_DIR / f"best_{direction}_bahdanau_attention.pt"
    best_validation_loss = float("inf")
    epochs_without_improvement = 0

    print("\n" + "=" * 90)
    print(f"Training {data_bundle['spec']['label']}")
    print("=" * 90)

    for epoch in range(1, EPOCHS + 1):
        start_time = time.perf_counter()
        train_metrics = run_epoch(
            model,
            data_bundle["loaders"]["train"],
            criterion,
            target_vocab.pad_id,
            optimizer=optimizer,
            teacher_forcing_ratio=TEACHER_FORCING_RATIO,
        )
        validation_metrics = run_epoch(
            model,
            data_bundle["loaders"]["validation"],
            criterion,
            target_vocab.pad_id,
            optimizer=None,
            teacher_forcing_ratio=0.0,
        )

        history["train_loss"].append(train_metrics["loss"])
        history["validation_loss"].append(validation_metrics["loss"])
        history["train_token_accuracy"].append(train_metrics["token_accuracy"])
        history["validation_token_accuracy"].append(
            validation_metrics["token_accuracy"]
        )

        elapsed = time.perf_counter() - start_time
        history["epoch_seconds"].append(elapsed)
        print(
            f"Epoch {epoch:02d}/{EPOCHS} | {elapsed:6.1f}s | "
            f"train loss {train_metrics['loss']:.4f}, ppl {train_metrics['perplexity']:.2f}, "
            f"acc {train_metrics['token_accuracy']:.3f} | "
            f"val loss {validation_metrics['loss']:.4f}, "
            f"ppl {validation_metrics['perplexity']:.2f}, "
            f"acc {validation_metrics['token_accuracy']:.3f}"
        )

        if validation_metrics["loss"] < best_validation_loss:
            best_validation_loss = validation_metrics["loss"]
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "direction": direction,
                    "best_validation_loss": best_validation_loss,
                    "source_vocabulary": data_bundle["source_vocab"].id_to_token,
                    "target_vocabulary": target_vocab.id_to_token,
                    "model_config": {
                        "attention": "bahdanau_additive",
                        "embedding_dim": EMBEDDING_DIM,
                        "hidden_dim": HIDDEN_DIM,
                        "num_layers": NUM_LAYERS,
                        "dropout": DROPOUT,
                    },
                },
                checkpoint_path,
            )
            print(f"  Saved best checkpoint: {checkpoint_path}")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print("  Early stopping.")
                break

    checkpoint = load_local_checkpoint(checkpoint_path)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, history, checkpoint_path

## 16. Huấn luyện hai chiều

Cell này lần lượt huấn luyện hai mô hình. Mô hình tốt nhất theo validation loss được nạp lại; mô hình thứ nhất được chuyển về CPU trước khi huấn luyện mô hình thứ hai để tiết kiệm VRAM.

In [ ]:
trained_models: dict[str, Seq2SeqBahdanau] = {}
training_histories: dict[str, dict[str, list[float]]] = {}
checkpoint_paths: dict[str, Path] = {}

for direction in DIRECTION_SPECS:
    model, history, checkpoint_path = train_direction(direction)
    trained_models[direction] = model
    training_histories[direction] = history
    checkpoint_paths[direction] = checkpoint_path

## 17. Đồ thị train/validation loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for axis, direction in zip(axes, DIRECTION_SPECS):
    history = training_histories[direction]
    epochs = range(1, len(history["train_loss"]) + 1)
    axis.plot(epochs, history["train_loss"], marker="o", label="train")
    axis.plot(epochs, history["validation_loss"], marker="o", label="validation")
    axis.set_title(DIRECTION_SPECS[direction]["label"])
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Cross-entropy loss")
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()

## 18. Greedy decoding theo batch

Khi inference không có Teacher Forcing. Decoder bắt đầu bằng `<SOS>`, lấy token có xác suất lớn nhất ở mỗi bước và dừng tại `<EOS>` hoặc `MAX_DECODE_LENGTH`.

In [ ]:
@torch.inference_mode()
def greedy_decode_batch(
    model: Seq2SeqBahdanau,
    source: torch.Tensor,
    source_lengths: torch.Tensor,
    target_vocab: Vocabulary,
    max_length: int = MAX_DECODE_LENGTH,
) -> tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    encoder_outputs, hidden, cell = model.encoder(source, source_lengths)
    source_mask = source.ne(model.source_pad_id)
    batch_size = source.size(0)
    input_token = torch.full(
        (batch_size,), target_vocab.sos_id, dtype=torch.long, device=source.device
    )
    finished = torch.zeros(batch_size, dtype=torch.bool, device=source.device)
    generated_steps = []
    attention_steps = []

    for _ in range(max_length):
        logits, hidden, cell, attention_weights = model.decoder(
            input_token,
            hidden,
            cell,
            encoder_outputs,
            source_mask,
        )
        next_token = logits.argmax(dim=1)
        next_token = torch.where(
            finished,
            torch.full_like(next_token, target_vocab.eos_id),
            next_token,
        )
        generated_steps.append(next_token)
        attention_steps.append(attention_weights)
        finished |= next_token.eq(target_vocab.eos_id)
        input_token = next_token
        if bool(finished.all()):
            break

    return (
        torch.stack(generated_steps, dim=1),
        torch.stack(attention_steps, dim=1),
    )


def metric_text(token_ids: Sequence[int], vocabulary: Vocabulary) -> str:
    return " ".join(vocabulary.decode(token_ids))


@torch.inference_mode()
def evaluate_sacrebleu(
    direction: str,
) -> tuple[dict[str, float | str], list[str], list[str]]:
    data_bundle = direction_data[direction]
    target_vocab = data_bundle["target_vocab"]
    model = trained_models[direction].to(DEVICE)
    hypotheses: list[str] = []
    # Reference lấy trực tiếp từ token gốc của test, không decode qua vocabulary.
    # Nhờ vậy từ OOV trong reference không bị biến thành <UNK>.
    references = [
        " ".join(tokens)
        for tokens in data_bundle["datasets"]["test"].frame[
            data_bundle["spec"]["target_column"]
        ]
    ]

    progress = tqdm(
        data_bundle["loaders"]["test"],
        desc=f"BLEU {data_bundle['spec']['label']}",
        leave=False,
    )
    for batch in progress:
        source = batch["source"].to(DEVICE, non_blocking=True)
        generated, _ = greedy_decode_batch(
            model,
            source,
            batch["source_lengths"],
            target_vocab,
        )
        generated = generated.cpu()

        hypotheses.extend(
            metric_text(row.tolist(), target_vocab) for row in generated
        )
    if len(hypotheses) != len(references):
        raise RuntimeError(
            f"{direction}: {len(hypotheses)} hypotheses != {len(references)} references"
        )

    # Dữ liệu đã được tokenize bằng tokenizer riêng, nên SacreBLEU không tokenize lần hai.
    bleu_metric = BLEU(tokenize="none", lowercase=False, effective_order=True)
    bleu_score = bleu_metric.corpus_score(hypotheses, [references])
    result = {
        "direction": data_bundle["spec"]["label"],
        "SacreBLEU": bleu_score.score,
        "brevity_penalty": bleu_score.bp,
        "hypothesis_tokens": bleu_score.sys_len,
        "reference_tokens": bleu_score.ref_len,
        "signature": str(bleu_metric.get_signature()),
    }

    model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result, hypotheses, references

## 19. Đánh giá SacreBLEU trên test chính thức

Đây là **tokenized, case-sensitive corpus BLEU** (`tok:none`) vì notebook đã tokenize trước. Signature được in để kết quả có thể tái lập.

In [ ]:
bleu_results = {}
test_hypotheses = {}
test_references = {}

for direction in DIRECTION_SPECS:
    result, hypotheses, references = evaluate_sacrebleu(direction)
    bleu_results[direction] = result
    test_hypotheses[direction] = hypotheses
    test_references[direction] = references

bleu_results_df = pd.DataFrame(bleu_results.values())
display(
    bleu_results_df[
        [
            "direction",
            "SacreBLEU",
            "brevity_penalty",
            "hypothesis_tokens",
            "reference_tokens",
            "signature",
        ]
    ].style.format({"SacreBLEU": "{:.2f}", "brevity_penalty": "{:.4f}"})
)

## 20. Benchmark theo độ dài câu và lưu kết quả

Ba nhóm source length (`≤15`, `16–29`, `≥30`) cho thấy Attention thay đổi chất lượng câu dài như thế nào. CSV dùng cùng schema với notebook không Attention và Luong Attention.

In [ ]:
def subset_sacrebleu(
    hypotheses: Sequence[str], references: Sequence[str], indices: Sequence[int]
) -> float:
    if not indices:
        return float("nan")
    metric = BLEU(tokenize="none", lowercase=False, effective_order=True)
    selected_hypotheses = [hypotheses[index] for index in indices]
    selected_references = [references[index] for index in indices]
    return metric.corpus_score(selected_hypotheses, [selected_references]).score


benchmark_rows = []
for direction, spec in DIRECTION_SPECS.items():
    source_lengths = test_df[f"{spec['source_language']}_length"].tolist()
    short_indices = [i for i, length in enumerate(source_lengths) if length <= 15]
    medium_indices = [i for i, length in enumerate(source_lengths) if 16 <= length <= 29]
    long_indices = [i for i, length in enumerate(source_lengths) if length >= 30]
    benchmark_rows.append(
        {
            "model": "Seq2Seq LSTM + Bahdanau",
            "attention": "bahdanau_additive",
            "direction": spec["label"],
            "SacreBLEU": bleu_results[direction]["SacreBLEU"],
            "short_BLEU": subset_sacrebleu(
                test_hypotheses[direction], test_references[direction], short_indices
            ),
            "medium_BLEU": subset_sacrebleu(
                test_hypotheses[direction], test_references[direction], medium_indices
            ),
            "long_BLEU": subset_sacrebleu(
                test_hypotheses[direction], test_references[direction], long_indices
            ),
            "short_count": len(short_indices),
            "medium_count": len(medium_indices),
            "long_count": len(long_indices),
            "training_seconds": sum(training_histories[direction]["epoch_seconds"]),
            "epochs_trained": len(training_histories[direction]["train_loss"]),
            "best_validation_loss": min(
                training_histories[direction]["validation_loss"]
            ),
            "parameters": sum(
                parameter.numel()
                for parameter in trained_models[direction].parameters()
            ),
            "quick_run": QUICK_RUN,
            "train_pairs": len(train_df),
            "test_pairs": len(test_df),
        }
    )

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_path = BENCHMARK_DIR / "benchmark_bahdanau.csv"
benchmark_df.to_csv(benchmark_path, index=False, encoding="utf-8")
display(
    benchmark_df.style.format(
        {
            "SacreBLEU": "{:.2f}",
            "short_BLEU": "{:.2f}",
            "medium_BLEU": "{:.2f}",
            "long_BLEU": "{:.2f}",
            "training_seconds": "{:.1f}",
            "best_validation_loss": "{:.4f}",
        }
    )
)
print(f"Saved benchmark: {benchmark_path}")

## 21. So sánh các benchmark đã chạy

Chạy notebook 01, 02 và 03 với cùng `QUICK_RUN` theo thứ tự để bảng có đủ ba mô hình. Cell tự bỏ kết quả khác số lượng train/test nhằm tránh một phép so sánh không công bằng.

In [ ]:
benchmark_files = sorted(BENCHMARK_DIR.glob("benchmark_*.csv"))
comparison_frames = [pd.read_csv(path) for path in benchmark_files]
all_benchmarks_df = (
    pd.concat(comparison_frames, ignore_index=True)
    if comparison_frames
    else pd.DataFrame()
)

if all_benchmarks_df.empty:
    print("No benchmark files found yet.")
    compatible_benchmarks_df = all_benchmarks_df
else:
    compatible_mask = (
        all_benchmarks_df["train_pairs"].eq(len(train_df))
        & all_benchmarks_df["test_pairs"].eq(len(test_df))
        & all_benchmarks_df["quick_run"]
        .astype(str)
        .str.lower()
        .eq(str(QUICK_RUN).lower())
    )
    compatible_benchmarks_df = all_benchmarks_df.loc[compatible_mask].copy()
    display(
        compatible_benchmarks_df.sort_values(["direction", "SacreBLEU"], ascending=[True, False])
        .style.format(
            {
                "SacreBLEU": "{:.2f}",
                "short_BLEU": "{:.2f}",
                "medium_BLEU": "{:.2f}",
                "long_BLEU": "{:.2f}",
                "training_seconds": "{:.1f}",
            }
        )
    )

    available_models = compatible_benchmarks_df["model"].nunique()
    if available_models < 3:
        print(
            f"Currently found {available_models}/3 compatible models. "
            "Run the remaining notebook(s) with the same configuration."
        )

    for direction_label in compatible_benchmarks_df["direction"].unique():
        direction_comparison = compatible_benchmarks_df.loc[
            compatible_benchmarks_df["direction"] == direction_label
        ].sort_values("model")
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        direction_comparison.plot(
            x="model",
            y=["SacreBLEU", "long_BLEU"],
            kind="bar",
            ax=axes[0],
            color=["#4C78A8", "#F58518"],
        )
        axes[0].set_title(f"Quality — {direction_label}")
        axes[0].set_ylabel("SacreBLEU")
        axes[0].tick_params(axis="x", rotation=20)
        direction_comparison.plot(
            x="model",
            y="training_seconds",
            kind="bar",
            legend=False,
            ax=axes[1],
            color="#54A24B",
        )
        axes[1].set_title(f"Training time — {direction_label}")
        axes[1].set_ylabel("Seconds")
        axes[1].tick_params(axis="x", rotation=20)
        plt.tight_layout()
        plt.show()

## 22. Dịch thử ít nhất 10 câu test chưa xuất hiện trong train

Cell kiểm tra cả câu nguồn tiếng Anh lẫn tiếng Việt không có trong train đã dùng. Sau đó hiển thị source, reference và dự đoán cho cả hai chiều.

In [ ]:
train_english_sentences = set(train_df["en"])
train_vietnamese_sentences = set(train_df["vi"])
unseen_test_mask = ~test_df["en"].isin(train_english_sentences) & ~test_df["vi"].isin(
    train_vietnamese_sentences
)
unseen_test_indices = test_df.index[unseen_test_mask].tolist()

if len(unseen_test_indices) < 10:
    raise ValueError("Fewer than 10 test pairs are unseen in the training set.")

# Chọn các câu ngắn trước để bảng kết quả dễ đọc, vẫn hoàn toàn thuộc test held-out.
display_indices = (
    test_df.loc[unseen_test_indices]
    .assign(total_length=lambda frame: frame["en_length"] + frame["vi_length"])
    .sort_values(["total_length", "line_number"])
    .head(10)
    .index.tolist()
)


def detokenize_for_display(tokenized_text: str, language: str) -> str:
    text = tokenized_text
    text = re.sub(r"\s+([.,!?;:%)\]}])", r"\1", text)
    text = re.sub(r"([(\[{])\s+", r"\1", text)
    if language == "en":
        text = re.sub(r"\s+(['’](?:s|re|ve|ll|d|m|t))\b", r"\1", text)
    return text.replace("_", " ") if language == "vi" else text


for direction, spec in DIRECTION_SPECS.items():
    rows = []
    for index in display_indices:
        if spec["source_language"] == "en":
            source = test_df.loc[index, "en"]
            reference = test_df.loc[index, "vi"]
        else:
            source = test_df.loc[index, "vi"]
            reference = test_df.loc[index, "en"]

        prediction = detokenize_for_display(
            test_hypotheses[direction][index], spec["target_language"]
        )
        rows.append(
            {
                "test_index": index,
                "source": source,
                "reference": reference,
                "prediction": prediction,
                "source_seen_in_train": source
                in (
                    train_english_sentences
                    if spec["source_language"] == "en"
                    else train_vietnamese_sentences
                ),
            }
        )

    print("\n" + "=" * 100)
    print(spec["label"])
    display(pd.DataFrame(rows))

## 23. Hàm dịch câu mới và trả về trọng số Attention

In [ ]:
@torch.inference_mode()
def translate_sentence(
    sentence: str,
    direction: str,
    return_attention: bool = False,
) -> str | dict:
    if direction not in DIRECTION_SPECS:
        raise ValueError(f"direction must be one of {list(DIRECTION_SPECS)}")

    spec = DIRECTION_SPECS[direction]
    source_language = spec["source_language"]
    target_language = spec["target_language"]
    source_vocab = VOCABS[source_language]
    target_vocab = VOCABS[target_language]

    normalized = normalize_text(sentence)
    source_tokens = TOKENIZERS[source_language](normalized)
    if not source_tokens:
        raise ValueError("The input sentence is empty after preprocessing.")
    if len(source_tokens) > MAX_SENTENCE_TOKENS:
        raise ValueError(
            f"Input has {len(source_tokens)} tokens; maximum is {MAX_SENTENCE_TOKENS}."
        )

    source_ids = torch.tensor(
        [source_vocab.encode(source_tokens)], dtype=torch.long, device=DEVICE
    )
    source_lengths = torch.tensor([source_ids.size(1)], dtype=torch.long)
    model = trained_models[direction].to(DEVICE)
    generated_batch, attention_batch = greedy_decode_batch(
        model,
        source_ids,
        source_lengths,
        target_vocab,
    )
    generated = generated_batch[0].cpu().tolist()
    prediction_tokens = target_vocab.decode(generated)
    prediction = detokenize_for_display(" ".join(prediction_tokens), target_language)

    target_attention_labels = []
    used_steps = 0
    for token_id in generated:
        token = target_vocab.id_to_token[int(token_id)]
        target_attention_labels.append(token)
        used_steps += 1
        if token == EOS_TOKEN:
            break

    attention_result = {
        "translation": prediction,
        "source_tokens": [SOS_TOKEN, *source_tokens, EOS_TOKEN],
        "target_tokens": target_attention_labels,
        "attention": attention_batch[
            0, :used_steps, : source_ids.size(1)
        ].cpu().numpy(),
    }

    model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return attention_result if return_attention else prediction

## 24. Kiểm tra padding mask và trực quan hóa ma trận Bahdanau Attention

Trước khi vẽ, cell xác nhận tổng trọng số tại mọi source `<PAD>` xấp xỉ 0. Trục ngang là source, trục dọc là token Decoder sinh ra; mỗi hàng có tổng bằng 1.

In [ ]:
@torch.inference_mode()
def verify_attention_padding_mask(direction: str) -> float:
    data_bundle = direction_data[direction]
    batch = next(iter(data_bundle["loaders"]["test"]))
    source = batch["source"].to(DEVICE)
    model = trained_models[direction].to(DEVICE)
    _, attention = greedy_decode_batch(
        model,
        source,
        batch["source_lengths"],
        data_bundle["target_vocab"],
        max_length=8,
    )
    padding_positions = source.eq(data_bundle["source_vocab"].pad_id)
    expanded_padding = padding_positions.unsqueeze(1).expand_as(attention)
    maximum_padded_weight = (
        float(attention.masked_select(expanded_padding).max().item())
        if bool(expanded_padding.any())
        else 0.0
    )
    model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if maximum_padded_weight > 1e-7:
        raise AssertionError(
            f"Attention mask failed: max PAD weight = {maximum_padded_weight}"
        )
    return maximum_padded_weight


for direction, spec in DIRECTION_SPECS.items():
    max_pad_weight = verify_attention_padding_mask(direction)
    print(f"{spec['label']}: maximum attention weight on <PAD> = {max_pad_weight:.2e}")


def plot_attention_heatmap(attention_result: dict, title: str) -> None:
    matrix = attention_result["attention"]
    source_labels = attention_result["source_tokens"]
    target_labels = attention_result["target_tokens"]
    if matrix.size == 0 or not target_labels:
        print(f"Cannot plot an empty translation: {title}")
        return

    figure_width = max(8.0, 0.42 * len(source_labels))
    figure_height = max(4.0, 0.34 * len(target_labels))
    fig, axis = plt.subplots(figsize=(figure_width, figure_height))
    image = axis.imshow(matrix, aspect="auto", cmap="viridis", vmin=0.0, vmax=1.0)
    axis.set_xticks(range(len(source_labels)))
    axis.set_xticklabels(source_labels, rotation=60, ha="right")
    axis.set_yticks(range(len(target_labels)))
    axis.set_yticklabels(target_labels)
    axis.set_xlabel("Source tokens")
    axis.set_ylabel("Generated target tokens")
    axis.set_title(title)
    fig.colorbar(image, ax=axis, label="Attention weight")
    plt.tight_layout()
    plt.show()


attention_example_indices = display_indices[:2]
for direction, spec in DIRECTION_SPECS.items():
    for test_index in attention_example_indices:
        source_sentence = test_df.loc[
            test_index, "en" if spec["source_language"] == "en" else "vi"
        ]
        attention_result = translate_sentence(
            source_sentence, direction, return_attention=True
        )
        plot_attention_heatmap(
            attention_result,
            f"Bahdanau — {spec['label']} — test #{test_index}",
        )

## 25. Dịch các câu tự tạo không có trong train

Các assertion xác nhận chuỗi token hóa của từng câu nguồn không xuất hiện trong tập train đã dùng.

In [ ]:
custom_examples = {
    "en_to_vi": [
        "Machine learning helps students explore new ideas.",
        "My robot is learning to translate short sentences.",
        "We will visit the science museum next Saturday.",
    ],
    "vi_to_en": [
        "Hôm nay tôi thử xây dựng một hệ thống dịch máy đơn giản.",
        "Nhóm sinh viên đang chuẩn bị cho cuộc thi trí tuệ nhân tạo.",
        "Chiếc máy tính mới chạy nhanh hơn tôi mong đợi.",
    ],
}

custom_translation_rows = []
for direction, sentences in custom_examples.items():
    spec = DIRECTION_SPECS[direction]
    source_language = spec["source_language"]
    source_column = spec["source_column"]
    training_source_keys = {
        " ".join(tokens) for tokens in train_df[source_column]
    }

    for sentence in sentences:
        normalized = normalize_text(sentence)
        tokens = TOKENIZERS[source_language](normalized)
        source_key = " ".join(tokens)
        assert source_key not in training_source_keys, (
            f"Custom sentence unexpectedly appears in train: {sentence}"
        )
        custom_translation_rows.append(
            {
                "direction": spec["label"],
                "source": sentence,
                "prediction": translate_sentence(sentence, direction),
                "seen_in_train": False,
            }
        )

display(pd.DataFrame(custom_translation_rows))

## 26. Mở rộng tùy chọn sang Việt–Trung

Dataset [`ngocdang83/tran-vi-teacher`](https://huggingface.co/datasets/ngocdang83/tran-vi-teacher) dùng schema `source_zh`/`target_vi`, yêu cầu chấp nhận điều kiện truy cập và phần lớn là đoạn văn dài. Khi mở rộng cần đăng nhập Hugging Face, tách đoạn thành câu, thêm tokenizer tiếng Trung (character hoặc Jieba), rồi huấn luyện một benchmark riêng. Không trộn điểm của dataset này vào bảng IWSLT'15 vì khác ngôn ngữ, domain và độ dài.

## 27. Kết luận

Notebook đã hoàn thành pipeline hai chiều, leakage-safe:

- kiểm tra số dòng và UTF-8 của sáu file song song;
- chuẩn hóa Unicode/HTML, lọc câu lỗi, câu dài, tỷ lệ độ dài bất thường và duplicate;
- tokenizer/vocabulary riêng cho English và Vietnamese, vocabulary chỉ học từ train;
- thêm `<PAD>`, `<SOS>`, `<EOS>`, `<UNK>` và padding động theo batch;
- cài đặt Bahdanau additive attention ở từng bước Decoder và masking `<PAD>`;
- huấn luyện hai Encoder–Decoder LSTM bằng Teacher Forcing, gradient clipping, checkpoint và early stopping;
- greedy decoding autoregressive trên câu held-out và câu tự tạo;
- trực quan hóa attention heatmap và kiểm thử trọng số tại padding bằng 0;
- đánh giá SacreBLEU toàn tập, theo độ dài câu, thời gian train và xuất benchmark chung.

So với baseline không Attention, Bahdanau dùng một mạng cộng phi tuyến để chấm điểm từng source state. Cơ chế này linh hoạt nhưng thêm phép chiếu và `tanh` tại mọi bước giải mã, vì vậy thường tốn thời gian hơn.